# Medical Context: Understanding Skin Lesions in HAM10000

## 1. Introduction: The Importance of Dermatoscopy

Melanoma is the deadliest form of skin cancer. Early detection is crucial, as the 5-year survival rate drops significantly once the disease metastasizes. Dermatoscopy, a non-invasive imaging technique, allows dermatologists to visualize subsurface skin structures not visible to the naked eye. This dataset, **HAM10000** (Human Against Machine with 10000 training images), is a benchmark for training AI to assist in this diagnosis.

## 2. The ABCD(E) Rule of Dermatoscopy

Dermatologists often use the **ABCDE** rule to evaluate potential melanomas. AI models implicitly learn features relevant to these criteria:

- **A - Asymmetry:** One half of the mole does not match the other.
- **B - Border:** Edges are irregular, ragged, notched, or blurred.
- **C - Color:** Color is not uniform (shades of brown, black, sometimes red, white, or blue).
- **D - Diameter:** The spot is larger than 6mm (pencil eraser size), though they can be smaller.
- **E - Evolving:** The mole is changing in size, shape, or color.

## 3. The 7 Diagnostic Categories in HAM10000

Our model is trained to classify lesions into one of these 7 categories. Understanding their pathology is key to interpreting model errors.

### 3.1. Melanocytic nevi (nv)
**Pathology:** Benign proliferations of melanocytes (pigment-producing cells). These are common moles.  
**Visuals:** Usually uniform in color and symmetric.  
**AI Challenge:** High intra-class variance (many shapes/sizes) but generally distinct from irregular carcinomas.

### 3.2. Melanoma (mel)
**Pathology:** Malignant tumor of melanocytes. The most dangerous class.  
**Visuals:** Often showcases the ABCDE features strongly (Asymmetry, irregular Border, multiple Colors).  
**AI Challenge:** Early-stage melanoma can look very similar to benign nevi, leading to false negatives.

### 3.3. Benign keratosis-like lesions (bkl)
**Pathology:** Includes seborrheic keratoses ("senile warts") and solar lentigines (liver spots).  
**Visuals:** "Stuck-on" appearance, waxy texture.  
**AI Challenge:** Textural features are important here.

### 3.4. Basal cell carcinoma (bcc)
**Pathology:** Common skin cancer arising from basal cells. Detailed by pearly papules or telangiectasia (visible small blood vessels).  
**Visuals:** Often pinkish, translucent, or pigmented.  
**AI Challenge:** Can be confused with vascular lesions or non-pigmented variations.

### 3.5. Actinic keratoses (akiec)
**Pathology:** Pre-cancerous (Bowen's disease) or intraepithelial carcinoma. Caused by sun damage.  
**Visuals:** Scaly, rough patches.  
**AI Challenge:** Can evolve into Squamous Cell Carcinoma.

### 3.6. Vascular lesions (vasc)
**Pathology:** Angiomas or angiokeratomas.  
**Visuals:** Red or purple color due to blood vessels (hemoglobin).  
**AI Challenge:** Distinct color features ease classification, making `vasc` often the easiest class for CNNs.

### 3.7. Dermatofibroma (df)
**Pathology:** Benign skin lesion likely due to insect bites or trauma.  
**Visuals:** Firm nodules, often with a central scar-like area.  

## 4. Dataset Distribution Analysis

Let's look at the actual distribution in our dataset. This reveals the **Imbalance Paradox**: accuracy is easy (just predict 'nevi'), but clinical value requires detecting rare 'melanoma'.

In [ ]:
import sys
import os
import seaborn as sns
import matplotlib.pyplot as plt

# Import our package
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from skin_cancer_detection import data, visualization

# Load Metadata
try:
    metadata = data.load_metadata()
    
    # Plot
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(x='diagnosis', data=metadata, order=metadata['diagnosis'].value_counts().index)
    plt.title('Class Imbalance in HAM10000')
    plt.xlabel('Diagnosis Class')
    plt.ylabel('Number of Samples')
    
    # annotating
    for p in ax.patches:
        ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', xytext = (0, 10), textcoords = 'offset points')
    plt.show()
except Exception as e:
    print(f"Could not load data for visualization: {e}")

**Observation:**  
`nv` (Nevi) dominates the dataset. This is why we use **Class Weighting** (mathematical penalty for ignoring rare classes) during training.